# QAOA optimization run on Quantinuum Nexus

**This notebook performs the *actual* variational QAOA optimization on the
Quantinuum Nexus cloud emulator** (not a functional-validation notebook -- for
that see `notebooks/validation.ipynb`).

The same Guppy kernel used locally (`src.qaoa.build_qaoa_instance`) is compiled to
HUGR and executed on Nexus's hosted **Selene emulator** (`SeleneConfig` +
`StatevectorSimulator`). A HUGR execute job returns a `QsysResult`, the *same* type
the local emulator returns, so energy decoding and result plotting are unchanged.

**Requirements:** network access and a Quantinuum Nexus account. Each optimizer
step submits one real cloud job, so a run of ~`maxiter` iterations submits that
many jobs sequentially. Keep `p`, `n_shots`, and `maxiter` modest.

## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import matplotlib.pyplot as plt

from src import qaoa_nexus as qn
from src.qaoa import brute_force_ground_state

## 2. Authenticate with Nexus\n\nOpens a browser to log in (token lasts ~30 days). Skip if already logged in.

In [ ]:
qn.login()

## 3. Load the cost Hamiltonian\n\nReads the diagonal `H_C` from `data/qubo_cr.json` (the same artifact the local solver uses).

In [ ]:
ch = qn.load_hamiltonian()
print(f"Cost Hamiltonian: {ch.n_qubits} qubits, "
      f"{len(ch.z_terms)} single-Z, {len(ch.zz_terms)} ZZ terms")
print("Variables:", ch.variables)

## 4. Configure the Nexus Selene emulator

`SeleneConfig` runs on Nexus's hosted Selene, using a seeded `StatevectorSimulator`
(the cloud twin of the local `Quest` engine) for reproducibility. Pass an
`error_model` (e.g. `DepolarizingErrorModel`) here to emulate hardware noise.

Setting the active project is required before submitting jobs -- `solve_scipy_nexus`
does this for us, but we create it explicitly so the config is visible.

In [ ]:
PROJECT = "kraken-quantathon"
SEED = 7

project = qn.get_project(PROJECT)
config = qn.selene_config(ch.n_qubits, seed=SEED)
print("Active project:", PROJECT)
print("Backend config:", config)

## 5. Run the variational optimization on Nexus

Each COBYLA evaluation compiles the kernel for the current angles, uploads the HUGR
package, submits an execute job to Nexus, waits for it, and decodes `<H_C>`. A live
callback prints each iteration's energy.

Tune the run below. Small values keep the number of cloud jobs (and wall-clock time)
low; increase for a more thorough search.

In [ ]:
P = 1            # QAOA layers
N_SHOTS = 500    # shots per circuit
MAXITER = 12     # COBYLA iterations (=> ~MAXITER cloud jobs)

trace = []
def show(it, energy):
    trace.append(energy)
    print(f"  iter {it:3d}   <H_C> = {energy:.4f}")

print(f"Running QAOA on Nexus (p={P}, shots={N_SHOTS}, maxiter={MAXITER})...")
result = qn.solve_scipy_nexus(
    ch, config=config, p_value=P, n_shots=N_SHOTS, seed=SEED,
    maxiter=MAXITER, project_name=PROJECT, progress=show,
)
print("\nDone.")

## 6. Optimization summary

In [ ]:
print(f"backend           = {result.metadata['backend']}")
print(f"optimizer         = {result.metadata['optimizer']}")
print(f"evaluations       = {result.metadata['n_evaluations']}")
print(f"converged         = {result.metadata['converged']}")
print(f"best <H_C>        = {result.energy:.4f}")
print(f"most-likely E     = {result.most_likely_energy():.4f}")
print(f"cost angles       = {np.round(result.cost_angles, 3)}")
print(f"mixer angles      = {np.round(result.mixer_angles, 3)}")

## 7. Convergence

In [ ]:
hist = result.metadata["history"]
plt.figure(figsize=(6, 4))
plt.plot(range(1, len(hist) + 1), hist, marker="o")
plt.xlabel("evaluation")
plt.ylabel(r"$\langle H_C \rangle$")
plt.title(f"QAOA on Nexus Selene (p={result.metadata['p']})")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Compare to the exact optimum\n\nThe subgraph is small enough to brute-force, so we can gauge how close the QAOA run got.

In [ ]:
gs_energy, gs_bits = brute_force_ground_state(ch)
gs_partition = dict(zip(ch.variables, gs_bits))
partition = result.most_likely_partition()

print(f"Brute-force optimum: E = {gs_energy:.4f}")
print(f"QAOA most-likely  E = {result.most_likely_energy():.4f}")
print(f"Exact match        : {result.most_likely_bits() == gs_bits}\n")

print("node -> side  (* = differs from optimum)")
for node, side in partition.items():
    mark = "" if partition[node] == gs_partition[node] else "  (*)"
    print(f"  {node}: {side}{mark}")

## 9. Paint the partition\n\nThe Nexus `QAOAResult` is the same type as local runs, so the standard painted-graph plot works directly.

In [ ]:
from IPython.display import Image
from src import qaoa

fig_path = qaoa.plot_partition(result, out="../figures/qaoa_partition_nexus.png",
                               label="QAOA fault-zone partition (Nexus Selene)")
print("Saved:", fig_path)
Image(str(fig_path))